In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pathlib
import yaml
import sys
sys.path.append("../")
from src.attentional_tracking_control_lightning import AttnTrackingControlModule

/home/jcruse/.conda/envs/jcruse_env/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
attn_path = "/om2/user/imgriff/projects/End-to-end-ASR-Pytorch/attn_cue_models/attn_cue_jsin_pilot_no_pretrain_bs_64_lr_1e-4/checkpoints/epoch=1-step=120790.ckpt"
ctrl_path_1 = "../multi_talker_control/jsin_precombined_gammatone_40_channels_20kHz_on_gpu_1e-4lr/checkpoints/epoch=5-step=741324.ckpt"
ctrl_path_2 = "../multi_talker_control/jsin_precombined_gammatone_40_channels_20kHz_on_gpu_1e-4lr/checkpoints/epoch=5-step=731324.ckpt"
checkpoint = torch.load(ctrl_path_2)

In [3]:
checkpoint['state_dict'].keys()

odict_keys(['model.cnn.layernorm0.weight', 'model.cnn.layernorm0.bias', 'model.cnn.conv0.weight', 'model.cnn.pooling0.hann_window2d', 'model.cnn.layernorm1.weight', 'model.cnn.layernorm1.bias', 'model.cnn.conv1.weight', 'model.cnn.pooling1.hann_window2d', 'model.cnn.layernorm2.weight', 'model.cnn.layernorm2.bias', 'model.cnn.conv2.weight', 'model.cnn.pooling2.hann_window2d', 'model.cnn.layernorm3.weight', 'model.cnn.layernorm3.bias', 'model.cnn.conv3.weight', 'model.cnn.pooling3.hann_window2d', 'model.cnn.layernorm4.weight', 'model.cnn.layernorm4.bias', 'model.cnn.conv4.weight', 'model.cnn.pooling4.hann_window2d', 'model.cnn.layernorm5.weight', 'model.cnn.layernorm5.bias', 'model.cnn.conv5.weight', 'model.cnn.pooling5.hann_window2d', 'model.cnn.layernorm6.weight', 'model.cnn.layernorm6.bias', 'model.cnn.conv6.weight', 'model.fc.weight', 'model.fc.bias', 'model.logits.weight', 'model.logits.bias'])

In [4]:
config_p = yaml.load(open("../config/attentional_cue/attn_tracking_control_high_snr.yaml", 'r'), Loader=yaml.FullLoader)

config_p

{'data': {'num_words': 794,
  'corpus': {'root': '/om2/user/msaddler/projects/ibmHearingAid/assets/data/datasets/JSIN_v3.00/nStim_20000/2000ms/rms_0.1/noiseSNR_-10_10/stimSR_20000/reverb_none/noise_all/JSIN_all_v3/subsets/'},
  'audio': {'rep_type': 'cochlea_filt',
   'rep_kwargs': {'sr': 20000,
    'env_sr': 8000,
    'n_channels': 40,
    'low_lim': 40,
    'use_pad': True,
    'rep_on_gpu': False,
    'env_extraction_type': 'Half-wave Rectification',
    'downsampling_type': 'TorchTransformsResample',
    'downsampling_kwargs': {'lowpass_filter_width': 64,
     'rolloff': 0.9475937167399596,
     'resampling_method': 'kaiser_window',
     'beta': 14.769656459379492}},
   'compression_type': 'coch_p3',
   'compression_kwargs': {'scale': 1,
    'offset': 1e-07,
    'clip_value': 5,
    'power': 0.3}},
  'loader': {'batch_size': 8}},
 'snr_condition': 'high',
 'noise_kwargs': {'low_snr': 5, 'high_snr': 10},
 'val_metric': 'ACC/val_acc',
 'model_name': 'AttnTrackingControl',
 'hparas': 

In [5]:
model = AttnTrackingControlModule.load_from_checkpoint(checkpoint_path=ctrl_path_1, config=config_p)

In [ ]:
ckpt = torch.load(ctrl_path_1, map_location=torch.device('cpu'))
model.load_state_dict(ckpt['state_dict'])